In [ ]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


In [ ]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver
!pip install -q rank_bm25
!pip install -q kiwipiepy

In [ ]:
# 모듈 불러오기
%run /content/AIFFEL_final_project_peekabook/research/src/state/state_v1.ipynb
%run /content/AIFFEL_final_project_peekabook/research/src/db/qdrant.py
%run /content/AIFFEL_final_project_peekabook/research/src/embedding/embedder.py

In [ ]:
import os
from google.colab import userdata

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Set the HCX API key environment variable
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')

# Set the Qdrant API key environment variable
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

# ── 초기화 ──────────────────────────────
embedder = LocalEmbedder("BAAI/bge-m3")   # 임베딩
db = QdrantDB(vector_size=1024)           # Qdrant 연결

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")     # OpenAI LLM
# llm = ChatClovaX(model="HCX-005", temperature=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Fusion RAG Node

**simple_rag_node** 대비 변경 사항:
- Qdrant에서 후보 풀(`POOL_SIZE`)을 넉넉하게 뽑은 뒤
- 그 후보들의 `book_intro`로 BM25 인덱스를 즉석 생성
- Vector 점수와 BM25 점수를 각각 Min-Max 정규화 후 가중 합산(`alpha`)
- 최종 상위 `TOP_K`개를 `retrieved_books`로 반환

| 하이퍼파라미터 | 기본값 | 설명 |
|---|---|---|
| `POOL_SIZE` | 20 | BM25 재정렬 대상 후보 수 |
| `TOP_K` | 5 | 최종 반환 도서 수 |
| `ALPHA` | 0.5 | Vector 비중 (1.0=Vector only, 0.0=BM25 only) |

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi
import numpy as np
import json


# ── 형태소 분석기 초기화 ─────────────────────
kiwi = Kiwi()

def tokenize(text: str) -> list[str]:
    """한국어 형태소 분석 후 명사/동사/형용사만 추출"""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.tag.startswith(("NN", "VV", "VA"))  # 명사, 동사, 형용사
    ]



# ── 하이퍼파라미터 ───────────────────────
POOL_SIZE = 20   # BM25 재정렬 대상 후보 수
TOP_K     = 5    # 최종 반환 도서 수
ALPHA     = 0.6  # Vector 비중 (0.0 ~ 1.0)


def fusion_rag_node(state: CRSState) -> dict:
    summary = state.get("summary", "")

    # ── 1. Vector 검색: 후보 풀 확보 ────────────────────────────
    query_vector = embedder.embed(summary)
    vector_results = db.search("books_v1", query_vector, limit=POOL_SIZE, threshold=0.3)

    if not vector_results:
        return {"retrieved_books": []}

    # ── 2. BM25 인덱스 생성 (후보 풀의 book_intro 대상) ─────────
    corpus_texts     = [r.payload.get("book_intro", "") for r in vector_results]
    tokenized_corpus = [tokenize(text) for text in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    print(bm25.idf)

    # ── 3. BM25 점수 계산 ────────────────────────────────────────
    tokenized_query = tokenize(summary)
    bm25_scores = bm25.get_scores(tokenized_query)           # shape: (POOL_SIZE,)

    # ── 4. Vector 점수 추출 ──────────────────────────────────────
    vector_scores = np.array([r.score for r in vector_results])  # cosine similarity

    # ── 5. Min-Max 정규화 ────────────────────────────────────────
    def normalize(scores: np.ndarray) -> np.ndarray:
        min_s, max_s = scores.min(), scores.max()
        if max_s - min_s < 1e-8:
            return np.zeros_like(scores)
        return (scores - min_s) / (max_s - min_s)

    norm_vector = normalize(vector_scores)
    norm_bm25   = normalize(bm25_scores)

    # ── 6. Fusion 점수 합산 ──────────────────────────────────────
    combined = ALPHA * norm_vector + (1 - ALPHA) * norm_bm25

    # ── 7. 상위 TOP_K 선택 ───────────────────────────────────────
    top_indices = np.argsort(combined)[::-1][:TOP_K]

    retrieved_books = [
        {
            "isbn":      vector_results[i].payload.get("isbn"),
            "title":     vector_results[i].payload.get("title"),
            "author":    vector_results[i].payload.get("author"),
            "book_intro": vector_results[i].payload.get("book_intro"),
        }
        for i in top_indices
    ]

    return {"retrieved_books": retrieved_books}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"소개: {b['book_intro']}"
        for b in books
    ])

    # 프롬프트 직접 정의
    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}}
]
"""
    )

    chain = rag_prompt | llm
    response = chain.invoke({
        "context": context,
        "summary": summary
    })

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        # 파싱 실패해도 LLM 응답 그대로 반환
        recommendations = response.content

    # recommendations 출력 확인
    # print(recommendations)

    return {
        "messages": [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

In [ ]:
# ── TEST 실행 ────────────────────────────────
graph_test = StateGraph(CRSState)
graph_test.add_node("rag", fusion_rag_node)  # simple_rag_node → fusion_rag_node
graph_test.add_node("llm", rag_llm_node)
graph_test.add_edge(START, "rag")
graph_test.add_edge("rag", "llm")
graph_test.add_edge("llm", END)
app_test = graph_test.compile()

# TEST_SUMMARY = "사용자는 수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례를 찾고 있으며, 역사 비문학 장르를 선호합니다. 천천히 깊이 있게 읽는 스타일로, 중요한 내용이나 흥미로운 부분을 메모하여 학생들에게 설명하는 데 활용하려고 합니다. 중급 난이도의 책을 선호하며, 학생들에게 긍정적인 영향을 줄 수 있는 내용을 중시하고 있습니다."
# TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."
TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."

result = app_test.invoke({
    "messages": [HumanMessage(content=TEST_SUMMARY)],
    "summary": TEST_SUMMARY,
    "retrieved_books": [],
    "recommendations": [],
})

{'문학을': 2.001480000210124, '전공하는': 2.5649493574615363, '대학생을': 2.5649493574615363, '비롯하여': 2.5649493574615363, '이제': 2.001480000210124, '막': 2.5649493574615363, '소설을': 0.38566248081198484, '배우는': 2.001480000210124, '고등학생,': 2.5649493574615363, '또': 2.5649493574615363, '좋아하는': 2.001480000210124, '일반': 2.5649493574615363, '교양인들이': 2.5649493574615363, '우리': 1.6094379124341005, '한국의': 1.2992829841302607, '현대소설': 2.001480000210124, '작품을': 0.38566248081198484, '올바르게': 2.5649493574615363, '읽고': 1.6094379124341005, '이해하는': 1.2992829841302607, '데': 1.2992829841302607, '하나의': 2.5649493574615363, '길잡이가': 2.5649493574615363, '될': 1.0360919316867756, '수': 0.6114665626547686, '있도록': 0.8023464725249374, '엮은': 2.5649493574615363, '시리즈이다.': 2.5649493574615363, '가운데': 2.001480000210124, '어떤': 2.5649493574615363, '가려': 2.5649493574615363, '읽을': 2.5649493574615363, '것인가,': 2.5649493574615363, '그리고': 1.0360919316867756, '어떻게': 2.001480000210124, '감상하고': 2.5649493574615363, '이해할': 1.2992829841302607, '것인가.'

In [ ]:
result["recommendations"]

[{'title': '우리 소설 어떻게 읽을 것인가 4',
  'author': '김병욱, 김춘섭, 정덕준 공',
  'isbn': '9788952303721',
  'reason': '이 책은 한국 현대소설을 올바르게 이해하고 감상하는 방법을 제시합니다. 다양한 작품 해설과 사고를 유도하는 문제들이 포함되어 있어, 감성을 풍부하게 하는 데 도움을 줄 것입니다.'},
 {'title': '관악산 뻐꾸기 - 문학대표소설선 4',
  'author': '김관식',
  'isbn': '9791191155600',
  'reason': '현대적인 감각의 소설로, 몽환적이고 재미있는 이야기 전개가 특징입니다. 감情에 집중하면서도 흥미를 잃지 않고 독서를 즐길 수 있는 작품입니다.'},
 {'title': '고전문학으로 만나는 한국어와 한국문화 - nk Korean 시리즈',
  'author': '박소연, 이명애',
  'isbn': '9791166850776',
  'reason': '고전 문학을 통해 한국의 문화와 감성을 접할 수 있어, 다양한 시각을 배우고자 하는 사용자의 목표에 부합합니다. 또한 중급 난이도의 작품들을 통해 감정 정리와 아름다움을 느낄 수 있는 기회를 제공합니다.'}]